In [5]:
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.eval.scorer import VolatilityScorer
from src.utils import test_toolkit as ts
from src.utils import ts_toolkit as tst

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

RESULTS_PATH = PROJECT_ROOT / "results"

In [7]:
csv_paths = [
    RESULTS_PATH / "garch" / "best_garch_predictions.csv",
    RESULTS_PATH / "egarch" / "best_egarch_predictions.csv",
    RESULTS_PATH / "har" / "best_har_predictions.csv",
    RESULTS_PATH / "wtd_har" / "best_wtd_har_predictions.csv",
    RESULTS_PATH / "gru" / "exec_2" / "gru_test_predictions.csv",
    RESULTS_PATH / "tcn" / "exec_6" / "tcn_test_predictions.csv",
    RESULTS_PATH / "bigru_tcn" / "exec_1" / "bigru_tcn_test_predictions.csv",
    RESULTS_PATH / "causal_transformer" / "exec_2" / "causal_transformer_test_predictions.csv",
]

model_names = [
    "GARCH",
    "EGARCH",
    "HAR",
    "WTD-HAR",
    "GRU",
    "TCN",
    "BiGRU-TCN",
    "Causal Transformer",
]

test_bundle = ts.load_test_predictions_matrix(
    csv_paths=csv_paths,
    model_names=model_names,
)

dates_test = test_bundle["dates"]
y_true_test = test_bundle["y_true"]
preds_matrix = test_bundle["preds_matrix"]
models_loaded = test_bundle["model_names"]

In [8]:
ts.display_models_comparison(
    y_true=y_true_test,
    preds_matrix=preds_matrix,
    model_names=models_loaded,
    metrics=("qlike", "crps", "pinball_05", "pinball_95", "rmse", "hmse", "hmae"),
    sort_by="qlike",
    decimals=4,
)

Modelo,QLIKE,CRPS,Pinball (τ=0.05),Pinball (τ=0.95),RMSE,HMSE,HMAE
TCN,0.3675,0.0123,0.0027,0.0015,0.0022,1.7030,0.6482
Causal Transformer,0.3936,0.0123,0.0027,0.0016,0.0021,2.1469,0.6853
WTD-HAR,0.4084,0.0124,0.0026,0.0015,0.0022,3.1898,0.7317
GARCH,0.4097,0.0123,0.0028,0.0016,0.0022,2.5417,0.6710
HAR,0.4471,0.0124,0.0026,0.0016,0.0022,3.3125,0.7970
EGARCH,0.5068,0.0123,0.0028,0.0017,0.0022,10.1008,0.7719
GRU,0.5109,0.0124,0.0027,0.0017,0.0020,6.4425,0.8131
BiGRU-TCN,0.6402,0.0126,0.0026,0.0020,0.0021,10.1471,1.0021


In [4]:
dm_results = ts.diebold_mariano_test(
    y_true=y_true_test,
    preds_matrix=preds_matrix,
    model_names=models_loaded,
    metric="qlike",
    alpha=0.05,
    n_bootstrap=1000,
    seed=SEED,
)

In [5]:
dm_results

{'metric': 'qlike',
 'models': ['GARCH', 'EGARCH', 'HAR', 'WTD-HAR', 'GRU', 'TCN'],
 'pairwise': {'GARCH_vs_EGARCH': {'model_a': 'GARCH',
   'model_b': 'EGARCH',
   'n': 953,
   'mean_loss_diff': -0.09714072855639606,
   'dm_stat': -1.3702782216971123,
   'p_value': 0.377,
   'significant': False,
   'alpha': 0.05,
   'ci_95_low': -0.25843108303164436,
   'ci_95_high': -0.0017182422933564266,
   'n_bootstrap': 1000,
   'block_len': 9,
   'bootstrap_type': 'stationary',
   'hac_lag': 7},
  'GARCH_vs_HAR': {'model_a': 'GARCH',
   'model_b': 'HAR',
   'n': 953,
   'mean_loss_diff': -0.03746294720107654,
   'dm_stat': -1.2788014602993336,
   'p_value': 0.249,
   'significant': False,
   'alpha': 0.05,
   'ci_95_low': -0.09750675364230385,
   'ci_95_high': 0.02225791919850492,
   'n_bootstrap': 1000,
   'block_len': 9,
   'bootstrap_type': 'stationary',
   'hac_lag': 7},
  'GARCH_vs_WTD-HAR': {'model_a': 'GARCH',
   'model_b': 'WTD-HAR',
   'n': 953,
   'mean_loss_diff': 0.00128076707881560